In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

embedding = model.encode("paracetamol is used for fever")

print(type(embedding))
print(embedding.shape)
print(embedding[:5])

/Users/maneeshari/Desktop/projects-learning/langchain-learning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|████████████| 103/103 [00:00<00:00, 8242.48it/s]


<class 'numpy.ndarray'>
(384,)
[-0.01558258 -0.03347386 -0.0396957  -0.03289833  0.01869687]


In [4]:
embedding1 = model.encode("paracetamol is used for fever")
embedding2 = model.encode("acetaminophen treats high temperature")
embedding3 = model.encode("robotics and sensors")

from numpy.linalg import norm
import numpy as np

def similarity(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

print(similarity(embedding1, embedding2))
print(similarity(embedding1, embedding3))

0.57129633
-0.003327854


In [5]:
embedding4 = model.encode("Tylenol reduces fever")
print(similarity(embedding1, embedding4))

0.64661133


In [6]:
import chromadb

client = chromadb.Client()
collection = client.create_collection("medicine")

print(collection)

Collection(name=medicine)


In [7]:
documents = [
    "paracetamol is used for fever and pain",
    "ibuprofen reduces inflammation and fever",
    "amoxicillin is an antibiotic for infections",
    "metformin is used to treat diabetes",
    "robotics and sensors are used in automation"
]

embeddings = [model.encode(doc).tolist() for doc in documents]

collection.add(
    documents = documents,
    embeddings = embeddings,
    ids = ["doc1", "doc2", "doc3", "doc4", "doc5"]
)

print("stored", collection.count(), "documents")

stored 5 documents


In [8]:
query =  "what medicine helps with fever?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings = [query_embedding],
    n_results = 2
)

print(results['documents'])

[['ibuprofen reduces inflammation and fever', 'paracetamol is used for fever and pain']]


In [12]:
import ollama

query = "what medicine helps with fever?"

query_embedding = model.encode(query).tolist()
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=2
)

retrieved_docs = results['documents'][0]
context = "\n".join(retrieved_docs)

response = ollama.chat(
    model="llama3.2",
    messages=[
        {"role": "user", "content": f"Answer this question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}"}
    ]
)

print(response['message']['content'])

According to the context, paracetamol is used for fever.
